In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/sales')

### **Drop Rescued Data Column**

In [0]:
df=df.drop("_rescued_data")

### **Checking Schema**

In [0]:
df.printSchema()

root
 |-- OrderDate: string (nullable = true)
 |-- StockDate: string (nullable = true)
 |-- OrderNumber: string (nullable = true)
 |-- ProductKey: string (nullable = true)
 |-- CustomerKey: string (nullable = true)
 |-- TerritoryKey: string (nullable = true)
 |-- OrderLineItem: string (nullable = true)
 |-- OrderQuantity: string (nullable = true)



### **Changing OrderDate and StockDate column's format**

In [0]:
df=df.withColumn('OrderDate',regexp_replace(col("OrderDate"),'/','-'))
df=df.withColumn('StockDate',regexp_replace(col("StockDate"),'/','-'))

In [0]:
df=df.withColumn('OrderDate',try_to_date(col("OrderDate"),'d-m-yyyy'))
df=df.withColumn('StockDate',try_to_date(col("StockDate"),'d-m-yyyy'))

### **Changing other column's format**

In [0]:
df=df.withColumn('ProductKey',col("ProductKey").cast('int'))\
    .withColumn('CustomerKey',col("CustomerKey").cast('int'))\
    .withColumn('TerritoryKey',col("TerritoryKey").cast('int'))\
    .withColumn('OrderLineItem',col("OrderLineItem").cast('int'))\
    .withColumn('OrderQuantity',col("OrderQuantity").cast('int'))

### **Fixing OrderNumber column's quality**

In [0]:
df=df.withColumn('OrderNumber',trim(col("OrderNumber")))

### **Checking Key column's correctness**

In [0]:
df_prd = spark.sql('''
                    select * from adventure_works.silver.products
                   ''')

In [0]:
df_cst = spark.sql('''
                    select * from adventure_works.silver.customers
                   ''')

In [0]:
df_terr = spark.sql('''
                    select * from adventure_works.silver.territories
                   ''')

In [0]:
df=df.filter((col('ProductKey') <= df_prd.select(max(col("ProductKey"))).collect()[0][0]) &
             (col('ProductKey') >= df_prd.select(min(col("ProductKey"))).collect()[0][0])
             )\
      .filter((col('CustomerKey') <= df_cst.select(max(col("CustomerKey"))).collect()[0][0]) &
             (col('CustomerKey') >= df_cst.select(min(col("CustomerKey"))).collect()[0][0])
             )\
      .filter((col('TerritoryKey') <= df_terr.select(max(col('SalesTerritoryKey'))).collect()[0][0]) &
             (col('TerritoryKey') >= df_terr.select(min(col("SalesTerritoryKey"))).collect()[0][0])
             )
      

### **Drop Duplicates**

In [0]:
df = df.dropDuplicates()

### **Business Logic**

In [0]:
df=df.filter(col("OrderDate") >= col("StockDate"))

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.sales'):
    df_silver_sales = spark.read.table('adventure_works.silver.sales')
    df = df.join(df_silver_sales,['OrderNumber','OrderLineItem'],'left_anti')

In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/sales')

In [0]:
%sql
create table if not exists adventure_works.silver.sales
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/sales'

In [0]:
df.display()

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity


In [0]:
%sql
select * from adventure_works.silver.sales

OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2017-01-01,2003-01-10,SO61269,229,11792,4,2,1
2017-01-01,2003-01-11,SO61272,480,21239,9,2,2
2017-01-01,2003-01-10,SO61276,488,17252,9,1,1
2017-01-01,2003-01-10,SO61274,475,27148,9,2,1
2017-01-01,2003-01-11,SO61309,354,13625,9,1,1
2017-01-01,2003-01-09,SO61302,477,14486,4,2,2
2017-01-01,2003-01-12,SO61299,528,12411,4,2,2
2017-01-01,2003-01-12,SO61297,530,12627,10,2,2
2017-01-01,2003-01-09,SO61293,220,18083,1,2,1
2017-01-01,2003-01-10,SO61287,478,15760,6,1,2
